# Entrenamiento reproducible y generacion de metricas

Notebook del proyecto **Panel de Domotica Controlado por Voz**.

Este notebook permite reproducir localmente el entrenamiento y la generacion de metricas de los dos modelos principales:

- Modelo base CNN robusto para comandos simples.
- Modelo secuencial GRU para comandos compuestos.

No usa APIs externas, servicios en la nube ni modelos preentrenados de reconocimiento de voz. Todo se ejecuta con el dataset local del proyecto.

## 1. Configuracion inicial

Las metricas generadas por este notebook se guardan en `metricas/notebook/` para no sobrescribir los resultados principales del repositorio.

In [ ]:
from pathlib import Path
import importlib.util
import os
import random
import sys

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

ROOT = Path.cwd()
if not (ROOT / "dataset_procesado").exists() and (ROOT.parent / "dataset_procesado").exists():
    ROOT = ROOT.parent

print(f"Python usado por este notebook: {sys.executable}")
print(f"Root del proyecto: {ROOT}")

required_modules = {
    "librosa": "librosa",
    "matplotlib": "matplotlib",
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "tensorflow": "tensorflow-cpu",
}
missing_modules = [module for module in required_modules if importlib.util.find_spec(module) is None]

if missing_modules:
    missing_packages = [required_modules[module] for module in missing_modules]
    requirements_path = ROOT / "requirements.txt"
    raise RuntimeError(
        "Faltan dependencias en el kernel actual: "
        + ", ".join(missing_packages)
        + "\n\nSolucion recomendada en VS Code/Jupyter:\n"
        + "1. Cambia el kernel del notebook al entorno .venv del proyecto.\n"
        + "2. Si aun faltan dependencias, ejecuta en PowerShell:\n"
        + f"   {sys.executable} -m pip install -r {requirements_path}\n\n"
        + "Si estas usando otro Python por accidente, instala el kernel del .venv con:\n"
        + "   .\\.venv\\Scripts\\python.exe -m pip install ipykernel\n"
        + "   .\\.venv\\Scripts\\python.exe -m ipykernel install --user --name voz-domotica --display-name \"Python (.venv voz-domotica)\""
    )

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.layers import (
    BatchNormalization,
    Bidirectional,
    Concatenate,
    Conv2D,
    Dense,
    Dropout,
    Flatten,
    GlobalAveragePooling1D,
    GlobalMaxPooling1D,
    GRU,
    Input,
    MaxPooling2D,
)
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.utils import to_categorical

DATASET_BASE_DIR = ROOT / "dataset_procesado" / "Base"
DATASET_SECUENCIAL_DIR = ROOT / "dataset_procesado" / "Secuencial"
MODEL_DIR = ROOT / "modelos"
METRICS_DIR = ROOT / "metricas" / "notebook"

MODEL_DIR.mkdir(exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)

SAMPLE_RATE = 16000
DURATION_SECONDS = 3.0
MAX_SAMPLES = int(SAMPLE_RATE * DURATION_SECONDS)
N_MFCC = 13
N_FFT = 512
HOP_LENGTH = 256
MAX_FRAMES = 188

TEST_SIZE = 0.20
VALIDATION_SPLIT = 0.15
BATCH_SIZE = 16
LEARNING_RATE = 0.001
EPOCHS_BASE = 100
EPOCHS_SECUENCIAL = 80

print(f"Root del proyecto: {ROOT}")
print(f"Metricas del notebook: {METRICS_DIR}")

## 2. Funciones de preprocesamiento MFCC

Se usa el mismo enfoque del proyecto: audio mono a 16 kHz, duracion fija de 3 segundos, 13 MFCC, `n_fft=512`, `hop_length=256` y 188 frames.

In [ ]:
def fix_audio_length(audio):
    audio = audio.astype(np.float32)
    if len(audio) > MAX_SAMPLES:
        return audio[:MAX_SAMPLES]
    if len(audio) < MAX_SAMPLES:
        return np.pad(audio, (0, MAX_SAMPLES - len(audio)), mode="constant")
    return audio


def fix_mfcc_cnn_frames(mfcc):
    current_frames = mfcc.shape[1]
    if current_frames > MAX_FRAMES:
        return mfcc[:, :MAX_FRAMES]
    if current_frames < MAX_FRAMES:
        return np.pad(mfcc, ((0, 0), (0, MAX_FRAMES - current_frames)), mode="constant")
    return mfcc


def fix_sequence_frames(mfcc_sequence):
    current_frames = mfcc_sequence.shape[0]
    if current_frames > MAX_FRAMES:
        return mfcc_sequence[:MAX_FRAMES, :]
    if current_frames < MAX_FRAMES:
        return np.pad(mfcc_sequence, ((0, MAX_FRAMES - current_frames), (0, 0)), mode="constant")
    return mfcc_sequence


def load_audio_fixed(file_path, trim_silence=False):
    audio, _ = librosa.load(Path(file_path), sr=SAMPLE_RATE, mono=True)
    audio = audio.astype(np.float32)
    if trim_silence and len(audio) > 0:
        trimmed_audio, _ = librosa.effects.trim(audio, top_db=30)
        if len(trimmed_audio) > 0:
            audio = trimmed_audio.astype(np.float32)
    return fix_audio_length(audio)


def audio_to_mfcc_cnn(audio):
    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=SAMPLE_RATE,
        n_mfcc=N_MFCC,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
    )
    mfcc = fix_mfcc_cnn_frames(mfcc)
    mfcc = (mfcc - np.mean(mfcc)) / (np.std(mfcc) + 1e-8)
    return np.expand_dims(mfcc, axis=-1).astype(np.float32)


def audio_to_mfcc_sequence(audio):
    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=SAMPLE_RATE,
        n_mfcc=N_MFCC,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
    )
    mfcc_sequence = fix_sequence_frames(mfcc.T)
    mfcc_sequence = (mfcc_sequence - np.mean(mfcc_sequence)) / (np.std(mfcc_sequence) + 1e-8)
    return mfcc_sequence.astype(np.float32)


def list_class_dirs(dataset_dir):
    dataset_dir = Path(dataset_dir)
    if not dataset_dir.exists():
        raise FileNotFoundError(f"No existe el dataset: {dataset_dir}")
    return sorted([path for path in dataset_dir.iterdir() if path.is_dir()])

## 3. Funciones para guardar metricas

Cada entrenamiento genera reporte de clasificacion, matriz de confusion, curvas de accuracy/loss y un resumen CSV.

In [ ]:
def save_history_plots(history, prefix):
    accuracy_path = METRICS_DIR / f"{prefix}_accuracy.png"
    loss_path = METRICS_DIR / f"{prefix}_loss.png"

    plt.figure()
    plt.plot(history.history["accuracy"], label="accuracy entrenamiento")
    plt.plot(history.history["val_accuracy"], label="accuracy validacion")
    plt.xlabel("Epoca")
    plt.ylabel("Accuracy")
    plt.title(f"Accuracy - {prefix}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(accuracy_path)
    plt.show()

    plt.figure()
    plt.plot(history.history["loss"], label="loss entrenamiento")
    plt.plot(history.history["val_loss"], label="loss validacion")
    plt.xlabel("Epoca")
    plt.ylabel("Loss")
    plt.title(f"Loss - {prefix}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(loss_path)
    plt.show()


def save_classification_outputs(y_true_labels, y_pred_labels, class_names, prefix):
    report = classification_report(
        y_true_labels,
        y_pred_labels,
        labels=class_names,
        target_names=class_names,
        zero_division=0,
    )
    report_path = METRICS_DIR / f"{prefix}_classification_report.txt"
    report_path.write_text(report, encoding="utf-8")

    cm = confusion_matrix(y_true_labels, y_pred_labels, labels=class_names)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    fig, ax = plt.subplots(figsize=(11, 9))
    disp.plot(cmap="Blues", xticks_rotation=45, ax=ax)
    ax.set_title(f"Matriz de confusion - {prefix}")
    fig.tight_layout()
    cm_path = METRICS_DIR / f"{prefix}_confusion_matrix.png"
    fig.savefig(cm_path)
    plt.show()

    print(report)
    print(f"Reporte guardado en: {report_path}")
    print(f"Matriz guardada en: {cm_path}")


def save_summary(prefix, test_loss, test_accuracy, model_path):
    summary_path = METRICS_DIR / f"{prefix}_summary.csv"
    pd.DataFrame(
        [
            {
                "modelo": prefix,
                "test_loss": float(test_loss),
                "test_accuracy": float(test_accuracy),
                "model_path": str(model_path.relative_to(ROOT)),
            }
        ]
    ).to_csv(summary_path, index=False)
    print(f"Resumen guardado en: {summary_path}")

## 4. Modelo base CNN robusto

Este bloque entrena el modelo para comandos simples usando `dataset_procesado/Base/`.

In [ ]:
EXPECTED_BASE_CLASSES = [
    "alarma",
    "apaga",
    "enciende",
    "puerta",
    "ruido_fondo",
    "seguro",
    "ventilador",
]


def collect_base_files():
    file_paths = []
    labels = []
    counts = {}
    for class_name in EXPECTED_BASE_CLASSES:
        class_dir = DATASET_BASE_DIR / class_name
        if not class_dir.exists():
            raise FileNotFoundError(f"Falta la carpeta de clase: {class_dir}")
        wav_files = sorted(class_dir.glob("*.wav"))
        counts[class_name] = len(wav_files)
        for wav_file in wav_files:
            file_paths.append(wav_file)
            labels.append(class_name)
    return np.array(file_paths, dtype=object), np.array(labels), counts


def augment_audio(audio, rng):
    augmented = []
    shift = int(rng.integers(-int(0.20 * SAMPLE_RATE), int(0.20 * SAMPLE_RATE) + 1))
    augmented.append(np.roll(audio, shift))
    augmented.append(np.clip(audio * float(rng.uniform(0.80, 1.20)), -1.0, 1.0))
    noise = rng.normal(0.0, float(rng.uniform(0.002, 0.008)), size=len(audio)).astype(np.float32)
    augmented.append(np.clip(audio + noise, -1.0, 1.0))
    augmented.append(librosa.effects.pitch_shift(y=audio, sr=SAMPLE_RATE, n_steps=float(rng.uniform(-1.0, 1.0))))
    augmented.append(librosa.effects.time_stretch(y=audio, rate=float(rng.uniform(0.90, 1.10))))
    return [item.astype(np.float32) for item in augmented]


def build_base_feature_set(file_paths, labels, augment=False):
    X = []
    y = []
    rng = np.random.default_rng(RANDOM_STATE)
    for file_path, label in zip(file_paths, labels):
        audio = load_audio_fixed(file_path, trim_silence=True)
        X.append(audio_to_mfcc_cnn(audio))
        y.append(label)
        if augment:
            for augmented_audio in augment_audio(audio, rng):
                X.append(audio_to_mfcc_cnn(fix_audio_length(augmented_audio)))
                y.append(label)
    return np.array(X, dtype=np.float32), np.array(y)


def build_base_cnn(input_shape, num_classes):
    model = Sequential(
        [
            Input(shape=input_shape),
            Conv2D(16, kernel_size=(3, 3), activation="relu", padding="same"),
            BatchNormalization(),
            MaxPooling2D(pool_size=(2, 2)),
            Dropout(0.20),
            Conv2D(32, kernel_size=(3, 3), activation="relu", padding="same"),
            BatchNormalization(),
            MaxPooling2D(pool_size=(2, 2)),
            Dropout(0.25),
            Conv2D(64, kernel_size=(3, 3), activation="relu", padding="same"),
            BatchNormalization(),
            MaxPooling2D(pool_size=(2, 2)),
            Dropout(0.30),
            Flatten(),
            Dense(64, activation="relu"),
            Dropout(0.30),
            Dense(num_classes, activation="softmax"),
        ]
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

In [ ]:
base_files, base_labels, base_counts = collect_base_files()
print("Cantidad de audios por clase base:")
for class_name, count in base_counts.items():
    print(f"- {class_name}: {count}")

train_files, test_files, train_labels, test_labels = train_test_split(
    base_files,
    base_labels,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=base_labels,
)

X_train_base, y_train_base_labels = build_base_feature_set(train_files, train_labels, augment=True)
X_test_base, y_test_base_labels = build_base_feature_set(test_files, test_labels, augment=False)

base_encoder = LabelEncoder()
base_encoder.fit(EXPECTED_BASE_CLASSES)
y_train_base = to_categorical(base_encoder.transform(y_train_base_labels), num_classes=len(base_encoder.classes_))
y_test_base = to_categorical(base_encoder.transform(y_test_base_labels), num_classes=len(base_encoder.classes_))

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(base_encoder.classes_)),
    y=base_encoder.transform(train_labels),
)
base_class_weight = {index: float(weight) for index, weight in enumerate(class_weights_array)}

print(f"Forma X_train_base: {X_train_base.shape}")
print(f"Forma X_test_base: {X_test_base.shape}")

In [ ]:
base_model = build_base_cnn(input_shape=X_train_base.shape[1:], num_classes=len(base_encoder.classes_))
base_model.summary()

base_callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)
]

base_history = base_model.fit(
    X_train_base,
    y_train_base,
    epochs=EPOCHS_BASE,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    callbacks=base_callbacks,
    class_weight=base_class_weight,
    verbose=2,
)

base_test_loss, base_test_accuracy = base_model.evaluate(X_test_base, y_test_base, verbose=0)
base_probabilities = base_model.predict(X_test_base, verbose=0)
base_pred_labels = base_encoder.inverse_transform(np.argmax(base_probabilities, axis=1))

base_model_path = MODEL_DIR / "notebook_modelo_base_cnn_robusto.keras"
base_classes_path = MODEL_DIR / "notebook_base_cnn_robusto_classes.npy"
base_model.save(base_model_path)
np.save(base_classes_path, base_encoder.classes_)

save_history_plots(base_history, "notebook_base_cnn_robusto")
save_classification_outputs(y_test_base_labels, base_pred_labels, base_encoder.classes_, "notebook_base_cnn_robusto")
save_summary("notebook_base_cnn_robusto", base_test_loss, base_test_accuracy, base_model_path)
print(f"Modelo guardado en: {base_model_path}")

## 5. Modelo secuencial GRU

Este bloque entrena el modelo para comandos compuestos usando `dataset_procesado/Secuencial/`.

In [ ]:
def load_sequential_dataset():
    X = []
    labels = []
    counts = {}
    for class_dir in list_class_dirs(DATASET_SECUENCIAL_DIR):
        wav_files = sorted(class_dir.glob("*.wav"))
        counts[class_dir.name] = len(wav_files)
        for wav_file in wav_files:
            audio = load_audio_fixed(wav_file, trim_silence=False)
            X.append(audio_to_mfcc_sequence(audio))
            labels.append(class_dir.name)
    return np.array(X, dtype=np.float32), np.array(labels), counts


def build_sequential_gru(num_classes):
    inputs = Input(shape=(MAX_FRAMES, N_MFCC))
    x = Bidirectional(GRU(64, return_sequences=True))(inputs)
    x = Dropout(0.20)(x)
    x = Bidirectional(GRU(32, return_sequences=True))(x)
    max_pool = GlobalMaxPooling1D()(x)
    avg_pool = GlobalAveragePooling1D()(x)
    x = Concatenate()([max_pool, avg_pool])
    x = Dense(96, activation="relu")(x)
    x = Dropout(0.20)(x)
    outputs = Dense(num_classes, activation="softmax")(x)
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

In [ ]:
X_seq, seq_labels, seq_counts = load_sequential_dataset()
print("Cantidad de audios por clase secuencial:")
for class_name, count in seq_counts.items():
    print(f"- {class_name}: {count}")

seq_encoder = LabelEncoder()
seq_encoded = seq_encoder.fit_transform(seq_labels)
seq_categorical = to_categorical(seq_encoded)

X_train_seq, X_test_seq, y_train_seq, y_test_seq = train_test_split(
    X_seq,
    seq_categorical,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=seq_encoded,
)

print(f"Forma X_train_seq: {X_train_seq.shape}")
print(f"Forma X_test_seq: {X_test_seq.shape}")

In [ ]:
seq_model = build_sequential_gru(num_classes=len(seq_encoder.classes_))
seq_model.summary()

seq_callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=12, restore_best_weights=True)
]

seq_history = seq_model.fit(
    X_train_seq,
    y_train_seq,
    epochs=EPOCHS_SECUENCIAL,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    callbacks=seq_callbacks,
    verbose=2,
)

seq_test_loss, seq_test_accuracy = seq_model.evaluate(X_test_seq, y_test_seq, verbose=0)
seq_probabilities = seq_model.predict(X_test_seq, verbose=0)
seq_pred_encoded = np.argmax(seq_probabilities, axis=1)
seq_true_encoded = np.argmax(y_test_seq, axis=1)
seq_pred_labels = seq_encoder.inverse_transform(seq_pred_encoded)
seq_true_labels = seq_encoder.inverse_transform(seq_true_encoded)

seq_model_path = MODEL_DIR / "notebook_modelo_secuencial_gru.keras"
seq_classes_path = MODEL_DIR / "notebook_secuencial_gru_classes.npy"
seq_model.save(seq_model_path)
np.save(seq_classes_path, seq_encoder.classes_)

save_history_plots(seq_history, "notebook_secuencial_gru")
save_classification_outputs(seq_true_labels, seq_pred_labels, seq_encoder.classes_, "notebook_secuencial_gru")
save_summary("notebook_secuencial_gru", seq_test_loss, seq_test_accuracy, seq_model_path)
print(f"Modelo guardado en: {seq_model_path}")

## 6. Comparacion final

Esta celda consolida los archivos `summary.csv` generados por ambos entrenamientos.

In [ ]:
summary_files = sorted(METRICS_DIR.glob("*_summary.csv"))
summary = pd.concat([pd.read_csv(path) for path in summary_files], ignore_index=True)
summary_path = METRICS_DIR / "resumen_entrenamientos_notebook.csv"
summary.to_csv(summary_path, index=False)
display(summary)
print(f"Resumen consolidado guardado en: {summary_path}")